# 01 — Generate TNG100-1 satellite catalogs

Builds the per-satellite catalogs used by `03_analysis.ipynb`, for Milky Way-mass hosts in
TNG100-1 (snapshot 99, z=0). This **consolidates** the two older notebooks
(`TNG100_SFMS_different_mass_selection`, `TNG100_SFMS_lower_satellite_mass`) into one
parameterized run and fixes the inconsistent little-h bookkeeping that those notebooks had.

**Runs only where the raw TNG100-1 catalogs are mounted (the Binder instance).** Requires
`illustris_python` and `h5py`.

For each satellite stellar-mass lower limit (10^6, 10^7, 10^8 M_sun, **physical**) it writes
`../data/tng100/tng100_satellites_<cut>.csv` with one row per selected satellite and **both**
mass conventions stored explicitly (see `../docs/conventions.md`):

* host selection (fiducial): `12.0 < log10(M_200c, physical) < 12.5`
* satellite selection: `M*,sat (physical) > cut`, plus host SDSS Sersic quality cuts.

> Little-h reminder: TNG masses are in `10^10 h^-1 M_sun`. Physical = `*1e10/h`;
> h^-1 = `*1e10`. We use physical for the science cuts (to match the SAGA/ELVES scale) and
> also store the h^-1 columns. (Karp+ 2023 instead works in h^-1 — see `02_tng_reproduce_karp`.)

In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import illustris_python as il
from scipy.stats import binned_statistic

## Configuration

In [ ]:
# --- simulation + redshift selection ---
# Pick TNG100 or TNG50 (differ only in directory name and box size; little-h is the same), and
# z=0 or z=0.05 (the two redshifts with SDSS-SKIRT mocks). The catalog/morphology layout is
# identical, so everything below is unchanged.
SIM      = 'TNG100'                 # 'TNG100' or 'TNG50'
REDSHIFT = 'z0'                     # 'z0' (z=0, snap 99) or 'z0p05' (z=0.05, snap 98)

_SIM_DIR    = {'TNG100': 'L75n1820TNG', 'TNG50': 'L35n2160TNG'}[SIM]
_LBOX_MPC_H = {'TNG100': 75.0,         'TNG50': 35.0}[SIM]
snap        = {'z0': 99, 'z0p05': 98}[REDSHIFT]
_Z          = {'z0': 0.0, 'z0p05': 0.0485}[REDSHIFT]    # redshift -> scale factor for phys. distances
_A          = 1.0 / (1.0 + _Z)                          # comoving -> physical (a=1 at z=0)

# TNG_ROOT must be an ABSOLUTE path to the simulation directory. On Binder the data sits under the
# home dir, i.e. ~/SimulationData/<sim_dir> -> /home/jovyan/SimulationData/<sim_dir>. Edit the
# parent below if yours lives elsewhere. (Absolute path avoids '/home/jovyanSimulationData/...'.)
TNG_ROOT = os.path.expanduser(f'~/SimulationData/{_SIM_DIR}')

_sp             = f'snapnum_{snap:03d}'
basePath        = os.path.join(TNG_ROOT, 'output')
morph_g         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_g.hdf5')
morph_i         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_i.hdf5')
subfind_id_path = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/subfind_ids.txt')

# fail early and clearly if the data is not where we think it is
_gc = os.path.join(basePath, f'groups_{snap:03d}', f'fof_subhalo_tab_{snap:03d}.0.hdf5')
assert os.path.isdir(basePath), f'basePath does not exist: {basePath}\n  -> edit TNG_ROOT for {SIM} ({_SIM_DIR}).'
assert os.path.exists(_gc), f'group catalog not found: {_gc}\n  -> check TNG_ROOT / snapshot {snap}.'
assert os.path.exists(morph_g), f'SDSS morphology not found: {morph_g}\n  -> z=0.05 mocks live at snap 98; check the path.'
print(f'SIM = {SIM} | REDSHIFT = {REDSHIFT} (snap {snap}) | TNG_ROOT = {TNG_ROOT}')

# --- cosmology / box ---
h    = 0.6774                       # IllustrisTNG little-h (Planck 2015)
Lbox = _LBOX_MPC_H * 1000.0 / h     # box side in physical kpc

# --- selections (FIDUCIAL = physical M_sun; see docs/conventions.md) ---
HOST_LOGM200_MIN = 12.0             # log10 M_200c [physical M_sun]
HOST_LOGM200_MAX = 12.5
SAT_MASS_CUTS    = 10.0 ** np.array([6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0])  # M*,sat lower limits, half-dex [physical M_sun]

# host SDSS-morphology quality cuts (Sersic fits must be reliable)
SN_MIN = 2.5

# restrict to satellites within R_200c of their host (matches the SAGA/ELVES within-R200c sample):
#   '3d' (default) = 3D separation, 'proj' = projected (x-y) separation, None = no radial cut
WITHIN_R200C = '3d'
R200C_FACTOR = 1.0

# --- output (per simulation AND redshift, so nothing overwrites) ---
OUTDIR = f'../data/{SIM.lower()}/{REDSHIFT}'
os.makedirs(OUTDIR, exist_ok=True)

## Load the group / subhalo catalogs and post-processing products

In [ ]:
# primary-halo (group) catalog
groups = il.groupcat.loadHalos(
    basePath, snap,
    fields=['GroupFirstSub', 'Group_M_Crit200', 'Group_R_Crit200', 'GroupNsubs'])

# subhalo catalog
subhalos = il.groupcat.loadSubhalos(
    basePath, snap,
    fields=['SubhaloGrNr', 'SubhaloMassType', 'SubhaloCM', 'SubhaloSFR'])

n_sub = len(subhalos['SubhaloGrNr'])

# host SDSS g/i Sersic morphologies (Rodriguez-Gomez+ 2019 SKIRT images)
sdss_g = h5py.File(morph_g, 'r')
sdss_i = h5py.File(morph_i, 'r')
subfind_ids = np.loadtxt(subfind_id_path)   # subhalo IDs the morphology rows correspond to

print('subhalos:', n_sub, '| groups:', len(groups['GroupFirstSub']))

## Select Milky Way-mass host halos and flag their satellites

`Group_M_Crit200` is in `10^10 h^-1 M_sun`; the **physical** host mass is `*1e10/h`.
Satellites of a group are the subhalos `GroupFirstSub+1 ... GroupFirstSub+GroupNsubs-1`
(index `GroupFirstSub` is the central). We build a boolean mask `sat_mask` over all subhalos
and, in the same pass, propagate each host's properties down to its satellites.

In [ ]:
# physical host halo mass, log10 M_sun (mass-less groups -> -inf, excluded by the host cut)
with np.errstate(divide='ignore'):
    host_logm200_phys = np.log10(groups['Group_M_Crit200'] * 1e10 / h)
host_sel = (host_logm200_phys > HOST_LOGM200_MIN) & (host_logm200_phys < HOST_LOGM200_MAX)

first_sub = groups['GroupFirstSub'][host_sel]   # central subhalo id of each selected host
n_subs    = groups['GroupNsubs'][host_sel]      # number of subhalos in each selected host
print('MW-mass hosts (12.0 < log10 M200c_phys < 12.5):', host_sel.sum())

# per-satellite arrays (indexed over ALL subhalos; filled only for satellites of selected hosts)
sat_mask        = np.zeros(n_sub, dtype=bool)
host_id_arr     = np.full(n_sub, -1, dtype=int)        # which selected host (0..Nhost-1)
host_center     = np.zeros((n_sub, 3))                 # host CM, physical kpc
host_r200_arr   = np.zeros(n_sub)                      # host R_200c, physical kpc
host_m200_phys  = np.zeros(n_sub)                      # log10 M_200c physical
host_m200_hinv  = np.zeros(n_sub)                      # log10 M_200c h^-1

host_m200_hinv_all = np.log10(groups['Group_M_Crit200'][host_sel] * 1e10)        # no /h
host_m200_phys_all = host_logm200_phys[host_sel]

for k in range(len(first_sub)):
    c   = first_sub[k]                 # central subhalo index
    sl  = slice(c + 1, c + n_subs[k])  # this host's satellites
    sat_mask[sl]       = True
    host_id_arr[sl]    = k
    host_center[sl]    = subhalos['SubhaloCM'][c] / h
    host_r200_arr[sl]  = groups['Group_R_Crit200'][host_sel][k] / h
    host_m200_phys[sl] = host_m200_phys_all[k]
    host_m200_hinv[sl] = host_m200_hinv_all[k]

print('satellite subhalos flagged:', sat_mask.sum())

## Propagate host Sersic orientation and quality flags to satellites

The host major-axis angle is the mean of the SDSS g- and i-band 2D Sersic position angles
(`sersic_theta`, radians → degrees). We also carry the reliability flags so we can require
`flag == 0`, `flag_sersic == 0`, `sn_per_pixel > 2.5` in both bands (Karp+ 2023, our draft).

In [ ]:
def host_morph_to_satellites(sdss, first_sub, n_subs, subfind_ids, n_sub):
    '''Look up each selected host's Sersic morphology and broadcast it to its satellites.'''
    theta = np.zeros(n_sub)   # host major-axis angle [deg]
    flag  = np.zeros(n_sub)   # 0 = ok
    sflag = np.zeros(n_sub)   # 0 = ok
    sn    = np.zeros(n_sub)   # S/N per pixel
    theta_all  = np.asarray(sdss['sersic_theta']) * 180.0 / np.pi
    flag_all   = np.asarray(sdss['flag'])
    sflag_all  = np.asarray(sdss['flag_sersic'])
    sn_all     = np.asarray(sdss['sn_per_pixel'])
    for k in range(len(first_sub)):
        c   = first_sub[k]
        row = np.where(subfind_ids == c)[0]
        if row.size == 0:        # host has no SKIRT morphology entry -> leave zeros, flag bad
            flag[c + 1:c + n_subs[k]] = 1
            continue
        r  = row[0]
        sl = slice(c + 1, c + n_subs[k])
        theta[sl] = theta_all[r]
        flag[sl]  = flag_all[r]
        sflag[sl] = sflag_all[r]
        sn[sl]    = sn_all[r]
    return theta, flag, sflag, sn

theta_g, flag_g, sflag_g, sn_g = host_morph_to_satellites(sdss_g, first_sub, n_subs, subfind_ids, n_sub)
theta_i, flag_i, sflag_i, sn_i = host_morph_to_satellites(sdss_i, first_sub, n_subs, subfind_ids, n_sub)

host_theta = 0.5 * (theta_g + theta_i)                      # mean g/i major-axis angle [deg]
host_good  = ((flag_g == 0) & (sflag_g == 0) & (sn_g > SN_MIN) &
              (flag_i == 0) & (sflag_i == 0) & (sn_i > SN_MIN))
print('satellites whose host has reliable g+i Sersic fits:', (sat_mask & host_good).sum())

## Projected azimuthal angle alpha (0 deg = major axis, 90 deg = minor axis)

Project onto the simulation x-y plane, apply periodic boundary conditions, take the satellite
azimuth `arctan2(dy, dx)`, and fold the difference from the host major axis into `[0, 90]`.

In [ ]:
def wrap_min_image(d, L):
    '''Minimum-image periodic wrap of a separation vector.'''
    return d - L * np.round(d / L)

def angle_from_major_axis(phi_sat_deg, pa_host_deg):
    '''Fold |phi_sat - PA_host| into [0, 90] using the disk's 180-deg / mirror symmetry.'''
    phi = np.mod(phi_sat_deg, 360.0)
    d1 = np.abs(phi - np.mod(pa_host_deg, 360.0))
    d2 = np.abs(phi - np.mod(pa_host_deg + 180.0, 360.0))
    d1 = np.minimum(d1, 360.0 - d1)
    d2 = np.minimum(d2, 360.0 - d2)
    m  = np.minimum(d1, d2)
    return np.minimum(m, 180.0 - m)

rel    = wrap_min_image(subhalos['SubhaloCM'] / h - host_center, Lbox)   # comoving kpc (=physical at z=0)
# projected azimuthal angle is measured IN THE x-y PLANE: arctan2(dy, dx)
phi_2d = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))                     # [-180, 180]
alpha  = angle_from_major_axis(phi_2d, host_theta)                       # [0, 90]

# host-centric distance: store BOTH the physical distance [kpc] and the ratio d/R_200c,
# each in 3D and projected onto the x-y plane. The ratio is frame-independent; the physical
# distance applies the scale factor _A (comoving -> physical; _A = 1 at z=0).
_d3d_com   = np.linalg.norm(rel, axis=1)                 # comoving kpc, 3D
_dproj_com = np.hypot(rel[:, 0], rel[:, 1])              # comoving kpc, projected (x-y)
d_3d_kpc   = _d3d_com   * _A                             # physical kpc, 3D
d_proj_kpc = _dproj_com * _A                             # physical kpc, projected
with np.errstate(invalid='ignore', divide='ignore'):
    _good_r     = host_r200_arr > 0
    d_r200_3d   = np.where(_good_r, _d3d_com   / host_r200_arr, np.nan)   # 3D distance / R_200c
    d_r200_proj = np.where(_good_r, _dproj_com / host_r200_arr, np.nan)   # projected distance / R_200c

## Satellite stellar mass (both conventions), SFR and quenched flag

Quenched = at least 1 dex below the SDSS SFMS fit of Martín-Navarro+ 2021,
`log10(SFR_MS) = 0.75*log10(M*) - 7.5`, using the **physical** stellar mass and `SubhaloSFR`.

In [ ]:
with np.errstate(divide='ignore'):   # star-less subhalos -> log10(0) = -inf (dropped by the mass cut)
    mstar_phys = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10 / h)   # physical M_sun
    mstar_hinv = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10)        # h^-1 M_sun
sat_sfr    = np.asarray(subhalos['SubhaloSFR'])

def quenched_flag(logmstar_phys, sfr):
    '''1 if >= 1 dex below the SFMS, else 0.'''
    sfr_ms = 10.0 ** (0.75 * logmstar_phys - 7.5)
    return (sfr < sfr_ms / 10.0).astype(int)

## Build and write one catalog per satellite mass cut

The selection per cut is: satellite of a MW-mass host, host has reliable g+i Sersic fits,
within R_200c (per `WITHIN_R200C`), and `M*,sat (physical) > cut`. We print self-checks and assert
the two mass conventions differ by `log10(1/h) = -log10(h) ~ +0.17 dex` before saving.

**On the overall f_q values:** do NOT expect these to equal Karp+'s reference f_q ~ 0.615.
That reference is for Karp's *full* halo-mass range (up to clusters) in *h^-1* units, whereas
this notebook is restricted to **MW-mass hosts (12.0-12.5)** in **physical** masses. Expect f_q to
*decrease* as the satellite mass cut increases (low-mass dwarfs quench more easily), and to be
generally lower than the cluster-inclusive value. The direct Karp comparison lives in
`02_tng_reproduce_karp.ipynb`. (At the 10^6 end, many low-mass subhalos have SubhaloSFR = 0 — a
TNG resolution floor — which counts them as quenched and lifts f_q; this matches the original
analysis.)

In [ ]:
# filename tag = log10 of the cut, e.g. 1e6 -> 'logM6.00', 1.58e7 -> 'logM7.20' (always unique)
def cut_tag(cut):
    return f'logM{np.log10(cut):.2f}'

# within-R_200c selection (applied to every cut below)
if WITHIN_R200C == '3d':
    radius_ok = d_r200_3d < R200C_FACTOR
elif WITHIN_R200C == 'proj':
    radius_ok = d_r200_proj < R200C_FACTOR
elif WITHIN_R200C in (None, 'none'):
    radius_ok = np.ones(len(sat_mask), dtype=bool)
else:
    raise ValueError("WITHIN_R200C must be '3d', 'proj', or None")
print(f'within-R200c cut: {WITHIN_R200C} (< {R200C_FACTOR} R200c)')

tags, logcuts, fq_list, nsat_list = [], [], [], []
for cut in SAT_MASS_CUTS:
    sel = sat_mask & host_good & radius_ok & (mstar_phys > np.log10(cut))

    df = pd.DataFrame({
        'host_id':        host_id_arr[sel],
        'mstar_phys':     mstar_phys[sel],
        'mstar_hinv':     mstar_hinv[sel],
        'host_m200_phys': host_m200_phys[sel],
        'host_m200_hinv': host_m200_hinv[sel],
        'sfr':            sat_sfr[sel],
        'alpha':          alpha[sel],
        'd_3d_kpc':       d_3d_kpc[sel],      # physical 3D host distance [kpc]
        'd_proj_kpc':     d_proj_kpc[sel],    # physical projected (x-y) host distance [kpc]
        'd_r200_3d':      d_r200_3d[sel],     # 3D distance / R_200c
        'd_r200_proj':    d_r200_proj[sel],   # projected distance / R_200c
        'quenched':       quenched_flag(mstar_phys[sel], sat_sfr[sel]),
    })

    # the two mass conventions must differ by log10(1/h) = -log10(h) ~ +0.17; angles in [0,90]; radius cut held
    assert np.allclose(df.mstar_phys - df.mstar_hinv, -np.log10(h), atol=1e-6), 'h-convention mismatch!'
    assert df.alpha.between(0, 90).all(), 'alpha outside [0, 90]!'
    if WITHIN_R200C == '3d':
        assert (df.d_r200_3d < R200C_FACTOR).all(), 'a satellite slipped past the 3D R200c cut!'

    tag = cut_tag(cut)
    tags.append(tag); logcuts.append(np.log10(cut))
    fq_list.append(df.quenched.mean()); nsat_list.append(len(df))
    out = f'{OUTDIR}/tng100_satellites_{tag}.csv'
    df.to_csv(out, index=False)
    print(f'logM*>{np.log10(cut):.2f}  N_sat={len(df):5d}  N_host={df.host_id.nunique():4d}  '
          f'f_q={df.quenched.mean():.3f}  -> {os.path.basename(out)}')

print(f'\nwrote {len(tags)} catalogs to {OUTDIR}')
# sanity: in MW-mass hosts, f_q should fall from the lowest to the highest mass cut
assert fq_list[0] > fq_list[-1], 'unexpected: f_q should be lower at the highest satellite mass cut'
print('OK: f_q decreases from the lowest to the highest satellite mass cut, as expected.')

## Sanity figures

Left: overall quenched fraction vs the satellite mass cut (should fall with increasing mass in
MW-mass hosts). Right: relative satellite count vs azimuthal angle for the lowest cut (best
statistics) — the major-axis excess near alpha ~ 0. The detailed Karp+ reproduction lives in
`02_tng_reproduce_karp.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

# f_q vs mass cut
axL.plot(logcuts, fq_list, 'o-', color='k')
axL.set_xlabel(r'$\log_{10}(M_{*,\rm sat}\ {\rm cut})\ [M_\odot]$')
axL.set_ylabel(r'overall quenched fraction $f_q$')
axL.set_ylim(0, 1)

# anisotropy for the lowest (best-sampled) cut
df_lo = pd.read_csv(f'{OUTDIR}/tng100_satellites_{tags[0]}.csv')
counts, edges = np.histogram(df_lo['alpha'], bins=20, range=(0, 90))
centers = 0.5 * (edges[:-1] + edges[1:])
axR.scatter(centers, counts / counts.mean(), color='k')
axR.axhline(1.0, ls='--', color='grey')
axR.set_xlabel(r'$\alpha$ from host major axis [deg]')
axR.set_ylabel('relative satellite count')
axR.set_xlim(0, 90)
axR.set_title(rf'$\log M_* > {logcuts[0]:.2f}$ (N={len(df_lo)})')
plt.tight_layout(); plt.show()